# Multimodal Hybrid RAG — Evaluation Notebook

Reproduces the RAGAS-style benchmark tables from the capstone specification:

- **Context Precision / Context Recall / Hit Rate @5** (and Faithfulness / Answer Relevance when an `OPENAI_API_KEY` is configured)
- **40 challenge questions**: 15 factual · 15 tabular · 10 visual
- **Hypothesis check**: hybrid multimodal system achieves **> 30% relative Context Recall gain** on tabular/chart questions vs. the naive text-only dense baseline

Run `mmrag eval` first — this notebook only renders `data/results/metrics.json` and `per_question.jsonl`.

In [ ]:
import json
from pathlib import Path
import pandas as pd

RES = Path('..') / 'data' / 'results'
metrics = json.loads((RES / 'metrics.json').read_text(encoding='utf-8'))
dataset = json.loads((RES / 'dataset.json').read_text(encoding='utf-8'))

print('backends:', metrics['backend'])
print('questions:', metrics['n_questions'], metrics['dataset_counts'])

In [ ]:
# Overall comparison table -------------------------------------------------
rows = []
for (key), v in metrics['aggregates'].items():
    system, cat = key.split('|')
    if cat != 'overall':
        continue
    rows.append({'System': system, 'n': v['n'],
                 'Context Precision': v['context_precision'],
                 'Context Recall': v['context_recall'],
                 'Hit Rate @5': v['hit_rate_at_5'],
                 'Faithfulness': v['faithfulness'],
                 'Answer Relevance': v['answer_relevance'],
                 'Mean latency (ms)': v['mean_latency_ms']})
pd.DataFrame(rows).set_index('System').style.format('{:.1%}', na_rep='n/a',
    subset=['Context Precision', 'Context Recall', 'Hit Rate @5'])

In [ ]:
# Per-category breakdown ---------------------------------------------------
rows = []
for key, v in metrics['aggregates'].items():
    system, cat = key.split('|')
    if cat == 'overall':
        continue
    rows.append({'System': system, 'Category': cat, 'n': v['n'],
                 'Context Recall': v['context_recall'],
                 'Hit Rate @5': v['hit_rate_at_5']})
pd.DataFrame(rows).style.format('{:.1%}',
    subset=['Context Recall', 'Hit Rate @5'])

In [ ]:
# Hypothesis check: >30% relative recall gain on tabular/chart questions ----
hyp = metrics['hypothesis']
print(json.dumps(hyp, indent=2))
assert 'supported' in hyp
if hyp.get('supported') is True:
    print('HYPOTHESIS SUPPORTED — hybrid clears the +30% relative recall bar.')
elif hyp.get('supported') is False:
    print('HYPOTHESIS NOT supported on this run.')
else:
    print('Hypothesis inconclusive:', hyp.get('note'))

In [ ]:
# Per-question drill-down: where did each system miss? ---------------------
pq = []
with (RES / 'per_question.jsonl').open(encoding='utf-8') as fh:
    for line in fh:
        pq.append(json.loads(line))
df = pd.DataFrame(pq)
print(df.columns.tolist())
df.head(10)

In [ ]:
# Misses (recall = 0) grouped by system/category --------------------------------
if {'system', 'category', 'context_recall'} <= set(df.columns):
    miss = df[df['context_recall'] == 0]
    print(miss.groupby(['system', 'category']).size().rename('misses'))
    miss[['system', 'category', 'question']].head(20)
else:
    print('columns available:', df.columns.tolist())